**Exploratory Data Analysis**

In [2]:
import pandas as pd

df = pd.read_csv("data/split_data_preprocessed/val_data_with_summaries.csv")
# Define the lengths for each hierarchy step
# 0->2 (Sector), 2->3 (Subsector), 3->4 (Group), 4->5 (Industry), 5->6 (National)
steps_config = [
    {'name': 'step1_sector', 'prev_len': 0, 'curr_len': 2},
    {'name': 'step2_subsector', 'prev_len': 2, 'curr_len': 3},
    {'name': 'step3_industry_group', 'prev_len': 3, 'curr_len': 4},
    {'name': 'step4_industry', 'prev_len': 4, 'curr_len': 5},
    {'name': 'step5_national_industry', 'prev_len': 5, 'curr_len': 6}
]

for step in steps_config:
    step_data = []
    for _, row in df.iterrows():
        code = str(row['NAICS_Code'])
        
        # Calculate 'prev' part (empty string for the first step)
        prev_code = code[:step['prev_len']] if step['prev_len'] > 0 else ""
        # Calculate 'next' part (the fuller code)
        next_code = code[:step['curr_len']]
        
        step_data.append({
            'inputs': row['inputs'],
            'prev': prev_code,
            'next': next_code
        })
    
    # Save each step to a separate file
    pd.DataFrame(step_data).to_csv(f"naics_{step['name']}.csv", index=False)

In [2]:
import pandas as pd

df = pd.read_csv("data/NAICS_data_with_websites.csv")

df['NAICS_Code'] = df['NAICS Code'].astype(str)

results = []

# Iterate through the hierarchy levels
# Level 2 (Sector), 3 (Subsector), 4 (Industry Group), 5 (Industry), 6 (National)
for length in range(2, 7):
    # Truncate the code to the specific length
    unique_codes = df['NAICS_Code'].str[:length].unique()
    
    results.append({
        'Digits': length,
        'Level_Name': ['Sector', 'Subsector', 'Industry Group', 'Industry', 'National Industry'][length-2],
        'Unique_Count': len(unique_codes)
    })

# Convert to DataFrame for a clean view
summary_df = pd.DataFrame(results)

print("--- NAICS Hierarchical Class Summary ---")
print(summary_df.to_string(index=False))

# Optional: Store these counts in a dictionary for your training loop
num_classes_dict = dict(zip(summary_df['Digits'], summary_df['Unique_Count']))
# Example usage: model = ModernBERTNAICSClassifier(n_classes=num_classes_dict[3])

--- NAICS Hierarchical Class Summary ---
 Digits        Level_Name  Unique_Count
      2            Sector            24
      3         Subsector            98
      4    Industry Group           309
      5          Industry           711
      6 National Industry          1066
